# LinkGuard Supply Chain Risk Prediction Model Training

This notebook demonstrates the training of machine learning models for supply chain risk prediction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import xgboost as xgb
import shap
import pickle
import sqlite3
import sys
sys.path.append('../utils')
from data_generator import generate_synthetic_data

## 1. Data Loading and Preparation

In [ ]:
# Generate synthetic data
suppliers_df, components_df, supply_links_df = generate_synthetic_data()

print("Data shapes:")
print(f"Suppliers: {suppliers_df.shape}")
print(f"Components: {components_df.shape}")
print(f"Supply Links: {supply_links_df.shape}")

# Display first few rows
suppliers_df.head()

## 2. Feature Engineering

In [ ]:
def engineer_features(df):
    """Engineer features for risk prediction"""
    
    # Financial stability (normalized revenue)
    df['financial_stability'] = df['revenue'] / df['revenue'].max()
    
    # Geographic risk (high-risk countries)
    high_risk_countries = ['China', 'Vietnam', 'Thailand']
    df['geo_risk'] = df['country'].isin(high_risk_countries).astype(int)
    
    # Ownership dependency risk
    df['dependency_risk'] = df['ownership_pct'].fillna(0) / 100.0
    
    # Reliability risk (inverse of reliability score)
    df['reliability_risk'] = 1 - df['reliability_score']
    
    # Capacity utilization (normalized)
    df['capacity_norm'] = df['capacity'] / df['capacity'].max()
    
    return df

# Apply feature engineering
suppliers_df = engineer_features(suppliers_df)

# Define features and target
feature_cols = ['financial_stability', 'geo_risk', 'dependency_risk', 'reliability_risk', 'capacity_norm']
X = suppliers_df[feature_cols].fillna(0)
y = (suppliers_df['risk_score'] > 0.6).astype(int)  # Binary classification: high risk vs low risk

print(f"Features shape: {X.shape}")
print(f"Target distribution: {y.value_counts()}")

## 3. Model Training

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# Train XGBoost model
xgb_model = xgb.XGBClassifier(random_state=42, max_depth=6, n_estimators=100)
xgb_model.fit(X_train, y_train)

print("Models trained successfully!")

## 4. Model Evaluation

In [ ]:
# Evaluate Random Forest
rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest Results:")
print(classification_report(y_test, rf_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_pred_proba):.3f}")

# Evaluate XGBoost
xgb_pred = xgb_model.predict(X_test)
xgb_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print("\nXGBoost Results:")
print(classification_report(y_test, xgb_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, xgb_pred_proba):.3f}")

## 5. Feature Importance Analysis

In [ ]:
# Plot feature importance for Random Forest
plt.figure(figsize=(10, 6))
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

sns.barplot(data=feature_importance, x='importance', y='feature')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. SHAP Analysis for Model Explainability

In [ ]:
# SHAP analysis for XGBoost model
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
plt.title('SHAP Feature Importance Summary')
plt.tight_layout()
plt.show()

## 7. Model Saving

In [ ]:
# Save the best performing model (XGBoost in this case)
model_path = '../models/risk_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(xgb_model, f)

print(f"Model saved to {model_path}")

# Save feature names for later use
feature_names_path = '../models/feature_names.pkl'
with open(feature_names_path, 'wb') as f:
    pickle.dump(feature_cols, f)

print(f"Feature names saved to {feature_names_path}")

## 8. Risk Prediction Examples

In [ ]:
# Example predictions on test data
sample_predictions = pd.DataFrame({
    'Supplier_ID': range(len(X_test)),
    'Actual_Risk': y_test.values,
    'Predicted_Risk_Prob': xgb_pred_proba,
    'Predicted_Risk_Class': xgb_pred
})

# Show top 10 highest risk predictions
top_risk = sample_predictions.nlargest(10, 'Predicted_Risk_Prob')
print("Top 10 Highest Risk Suppliers:")
print(top_risk)